In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


In [2]:
df = pd.read_csv('cleaned_emotions.csv') 
X = df['cleaned_text'] # features
y = df['emotion_encoded'] # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1599,)
X_test shape: (400,)
y_train shape: (1599,)
y_test shape: (400,)


In [3]:
cv = CountVectorizer() 

X_train_bow = cv.fit_transform(X_train) 
X_test_bow = cv.transform(X_test) 

print("X_train_bow shape:", X_train_bow.shape)
print("X_test_bow shape:", X_test_bow.shape)

print("First 20 feature names:")
print(cv.get_feature_names_out()[:20])

X_train_bow shape: (1599, 4088)
X_test_bow shape: (400, 4088)
First 20 feature names:
['aaaah' 'abandoned' 'abba' 'abilities' 'ability' 'abit' 'able' 'abou'
 'absence' 'absoloutely' 'absolutely' 'abuse' 'abused' 'abyss' 'accept'
 'acceptable' 'acceptance' 'accepted' 'accepting' 'access']


In [ ]:
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)

y_pred_bow = nb_bow.predict(X_test_bow)
acc_bow = accuracy_score(y_test, y_pred_bow)
print("Accuracy with BoW Unigrams:", acc_bow)

Accuracy with BoW Unigrams: 0.6175


In [9]:
cv_bigram = CountVectorizer(ngram_range=(1,2)) 

X_train_bigram = cv_bigram.fit_transform(X_train)
X_test_bigram = cv_bigram.transform(X_test)

print("Created! Shape:", X_train_bigram.shape)
print("First 10 features:", cv_bigram.get_feature_names_out()[:10])

Created! Shape: (1599, 15540)
First 10 features: ['aaaah' 'abandoned' 'abandoned little' 'abandoned seemed' 'abba'
 'abba pointing' 'abilities' 'abilities writer' 'ability' 'ability others']


In [10]:
nb_bigram = MultinomialNB()
nb_bigram.fit(X_train_bigram, y_train)

y_pred_bigram = nb_bigram.predict(X_test_bigram)
acc_bigram = accuracy_score(y_test, y_pred_bigram)
print("Accuracy with BoW Unigrams+Bigrams:", acc_bigram)
print("Comparison with Q3 Unigram accuracy:", acc_bow, "vs", acc_bigram)

Accuracy with BoW Unigrams+Bigrams: 0.62
Comparison with Q3 Unigram accuracy: 0.6175 vs 0.62


In [11]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train) 
X_test_tfidf = tfidf.transform(X_test)
print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

print("\nFirst 15 TF-IDF feature names:")
print(tfidf.get_feature_names_out()[:15])

X_train_tfidf shape: (1599, 4088)
X_test_tfidf shape: (400, 4088)

First 15 TF-IDF feature names:
['aaaah' 'abandoned' 'abba' 'abilities' 'ability' 'abit' 'able' 'abou'
 'absence' 'absoloutely' 'absolutely' 'abuse' 'abused' 'abyss' 'accept']


In [12]:
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)

print("Accuracy with TF-IDF:", acc_tfidf)

Accuracy with TF-IDF: 0.5775


In [13]:
results = pd.DataFrame({
    "Method": ["BoW Unigrams", "BoW Unigrams+Bigrams", "TF-IDF"],
    "Accuracy": [acc_bow, acc_bigram, acc_tfidf]
})

print(results)
print("\nBest Method:", results.loc[results['Accuracy'].idxmax(), "Method"])
print("Best Accuracy:", results['Accuracy'].max())

print("\nObservation:")
print("TF-IDF usually performs best because it gives less weight to very common words")
print("and more weight to words that are important for a specific emotion.")
print("Bigrams help capture phrases like 'not happy' which unigrams miss.")

                 Method  Accuracy
0          BoW Unigrams    0.6175
1  BoW Unigrams+Bigrams    0.6200
2                TF-IDF    0.5775

Best Method: BoW Unigrams+Bigrams
Best Accuracy: 0.62

Observation:
TF-IDF usually performs best because it gives less weight to very common words
and more weight to words that are important for a specific emotion.
Bigrams help capture phrases like 'not happy' which unigrams miss.


In [ ]:

accuracies = {"BoW": acc_bow, "Bigram": acc_bigram, "TF-IDF": acc_tfidf}
best_method = max(accuracies, key=accuracies.get)

print("All Accuracies:", accuracies)
print("Best method:", best_method)

if best_method == "TF-IDF":
    best_vectorizer = tfidf
    best_model = nb_tfidf
elif best_method == "Bigram":
    best_vectorizer = cv_bigram
    best_model = nb_bigram
else:
    best_vectorizer = cv
    best_model = nb_bow

joblib.dump(best_vectorizer, 'best_vectorizer.pkl')
joblib.dump(best_model, 'best_model.pkl')

print("\nSaved: best_vectorizer.pkl and best_model.pkl")

test_sentence = ["i am feeling very sad today"]
test_vec = best_vectorizer.transform(test_sentence)
prediction = best_model.predict(test_vec)
print("Test prediction for:", test_sentence[0])